In [87]:
from collections import defaultdict
import pandas as pd
import json

In [88]:
df = pd.read_csv("KoMa-test2.csv")

In [89]:
ak_names = [col_name for col_name in df.columns if "Unnamed" not in col_name]
assert len(ak_names) == len(set(ak_names)), "duplicate ak names"
print(ak_names)
df

['ak1', 'ak2']


,Unnamed: 0,ak1,ak2,Unnamed: 3
Lorenzo2,NaN,Nein,Nein,NaN
Test,NaN,Ja,Unter Vorbehalt,NaN
noch jemand,NaN,Ja,Ja,NaN
ein kömischer Name,NaN,Nein,Unter Vorbehalt,NaN


In [90]:
ak_name_to_id_dict = {
    "ak1": "ak_id1",
    "ak2": "ak_id2",
}

In [91]:
def calc_pref_score(entry: str) -> int:
    if entry in ["Nein", "No"]:
        return 0
    elif entry in ["Ja", "Yes"]:
        return 2
    elif entry in ["Unter Vorbehalt", "Under reserve"]:
        return 1
    else:
        raise NotImplementedError

In [92]:
result_dict = {"participants": []}

index_lst = []
for idx, entries in df.iterrows():
    participant_dict = {
        "id": idx,
        "info": {},
        "time_constraints": [],
        "room_constraints": [],
        "preferences": [],
    }

    for ak_name in ak_names:
        score = calc_pref_score(entries[ak_name])
        if score != 0:
            participant_dict["preferences"].append(
                {
                    "ak_id": ak_name_to_id_dict[ak_name],
                    "required": False,
                    "preference_score": score,
                }
            )
    if not participant_dict["preferences"]:
        continue
    result_dict["participants"].append(participant_dict)
    index_lst.append(idx)

assert len(index_lst) == len(set(index_lst)), "duplicate participant names"

In [93]:
result_dict

{'participants': [{'id': 'Test',
   'info': {},
   'time_constraints': [],
   'room_constraints': [],
   'preferences': [{'ak_id': 'ak_id1',
     'required': False,
     'preference_score': 2},
    {'ak_id': 'ak_id2', 'required': False, 'preference_score': 1}]},
  {'id': 'noch jemand',
   'info': {},
   'time_constraints': [],
   'room_constraints': [],
   'preferences': [{'ak_id': 'ak_id1',
     'required': False,
     'preference_score': 2},
    {'ak_id': 'ak_id2', 'required': False, 'preference_score': 2}]},
  {'id': 'ein kömischer Name',
   'info': {},
   'time_constraints': [],
   'room_constraints': [],
   'preferences': [{'ak_id': 'ak_id2',
     'required': False,
     'preference_score': 1}]}]}

In [94]:
with open("export.json", "w") as f:
    json.dump(result_dict, f, indent=4)